In [1]:
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch

In [2]:
model_name = 'qwen2vl'
prompt_type = 'zeroshot'

In [3]:
dataset = 'type1'
image_type = 'original'
device = "cuda:1"

In [4]:
qas = pd.read_json(f'../{dataset}/qa.json')
qas

,question_id,image_index,qa_id,qa_type,question,answer
0,0,5,qa_8,Original,Which year recorded the least daily hempseed p...,1990
1,1,5,qa_9,Original,Which of the following saw the higher daily he...,Europe
2,2,5,qa_1,LLM,What is the total hempseed production for the ...,The total hempseed production for the world in...
3,3,5,qa_3,LLM,What is the percentage increase in daily hemps...,The percentage increase in daily hempseed prod...
4,4,5,qa_5,LLM,What is the ratio of hempseed production in Eu...,The ratio is 0.236
...,...,...,...,...,...,...
2804,2804,900,qa_3,LLM,In which year was there the highest ratio of q...,2019
2805,2805,900,qa_11,Original,What is the maximum crime rate shown in the ch...,8000
2806,2806,900,qa_12,LLM,In which year did the most significant increas...,2019
2807,2807,900,qa_12,Original,In which year did the maximum crime rate occur?,2002


In [5]:
image_indices = qas['image_index'].values.astype(int)
questions = qas['question'].values
answers = qas['answer'].values

In [6]:
image_base_path = f'../{dataset}/{image_type}/'

In [7]:
all_images = os.listdir(image_base_path)
index_to_image = {}

if dataset == 'type1':
    prefix = 'multichart_'
    if image_type == 'original' or image_type == 'simple':
        sep = '_'
    else:
        sep = '.'

for index in set(image_indices):
    for image in all_images:
        if image.startswith(f'{prefix}{index}{sep}'):
            if index not in index_to_image:
                index_to_image[index] = []
            index_to_image[index].append(image)

index_to_image

{np.int64(5): ['multichart_5_orig.png'],
 np.int64(6): ['multichart_6_orig.png'],
 np.int64(7): ['multichart_7_orig.png'],
 np.int64(8): ['multichart_8_orig.png'],
 np.int64(9): ['multichart_9_orig.png'],
 np.int64(52): ['multichart_52_orig.png'],
 np.int64(53): ['multichart_53_orig.png'],
 np.int64(54): ['multichart_54_orig.png'],
 np.int64(55): ['multichart_55_orig.png'],
 np.int64(56): ['multichart_56_orig.png'],
 np.int64(57): ['multichart_57_orig.png'],
 np.int64(58): ['multichart_58_orig.png'],
 np.int64(59): ['multichart_59_orig.png'],
 np.int64(61): ['multichart_61_orig.png'],
 np.int64(62): ['multichart_62_orig.png'],
 np.int64(63): ['multichart_63_orig.png'],
 np.int64(64): ['multichart_64_orig.png'],
 np.int64(65): ['multichart_65_orig.png'],
 np.int64(66): ['multichart_66_orig.png'],
 np.int64(67): ['multichart_67_orig.png'],
 np.int64(68): ['multichart_68_orig.png'],
 np.int64(69): ['multichart_69_orig.png'],
 np.int64(70): ['multichart_70_orig.png'],
 np.int64(71): ['mult

In [8]:
message_template = [
    {
        "role": "user",
        "content": []
    }
]

In [9]:
def get_message(image_index, prompt, question):
    images = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        images.append({
            "type": "image",
            "image": image_path
        })
    message = message_template.copy()
    message[0]['content'] = images
    message[0]['content'].append({
        "type": "text",
        "text": prompt.format(question)
    })
    return message

In [10]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map=device,
    cache_dir='/media/vivek/drive/multichartqa/models_cache'
)
model.eval()
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}
You are attempting to use Flash Attention 2.0 without specifying a torch dtype. This might lead to unexpected behaviour
`Qwen2VLRotaryEmbedding` can now be fully parameterized by passing the model config through the `config` argument. All other arguments will be removed in v4.46


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
model_responses = []

In [12]:
def run_model():
    for i, question in enumerate(tqdm(questions)):
        image_index = image_indices[i]
        prompt = """Your task is the answer the question based on the given image. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {}""" 
        messages = get_message(image_index, prompt, question)
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(device)
        
        generated_ids = model.generate(
            **inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        model_responses.append({'question_id': i, 'question': question, 'gold': answers[i], 'response': output_text[0].strip()})

In [13]:
run_model()

100%|██████████| 2809/2809 [53:17<00:00,  1.14s/it]  


In [14]:
# saving the model responses
os.makedirs(f'../model_responses/{dataset}', exist_ok=True)

model_responses_df = pd.DataFrame(model_responses)
model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records')